In [6]:
import ee

GEE_PROJECT = "gen-lang-client-0412358476"
try:
    ee.Initialize(project=GEE_PROJECT)
except:
    ee.Authenticate()
    ee.Initialize(project=GEE_PROJECT)
print("EE initialized |", GEE_PROJECT)
import time

EE initialized | gen-lang-client-0412358476


In [7]:
# ============================================================
# CELL 0 — Setup dasar (jalankan pertama, tidak assume apa pun ada)
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

!pip install geemap contextily -q

import ee
ee.Authenticate()
ee.Initialize(project='gen-lang-client-0412358476')  # ganti sesuai project GEE kamu

import os, json, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests, io

MODEL_DIR = "/content/drive/MyDrive/Data_experiment_shoreline/models"
LOG_DIR   = "/content/drive/MyDrive/Data_experiment_shoreline/logs"
LANDSAT_MASK_DIR = "/content/drive/MyDrive/Data_experiment_shoreline/masks_landsat"
os.makedirs(LOG_DIR, exist_ok=True)
os.makedirs(LANDSAT_MASK_DIR, exist_ok=True)

PATCH_SIZE = 256
SCALE_M = 10
LOOKBACK = 3
# ============================================================
# CELL 1 — AOI_CONFIG (definisi eksplisit, tidak assume dari file lain)
# ============================================================
AOI_CONFIG = {
    "Titik_02": {"coord": [110.4666041, -5.7915568], "buffer_m": 1500, "note": "erosion signal"},
    "Titik_04": {"coord": [110.4796091, -5.7724370], "buffer_m": 1500, "note": ""},
    "Titik_05": {"coord": [110.4500360, -5.8134327], "buffer_m": 1500, "note": ""},
    "Titik_06": {"coord": [110.4802625, -5.7989676], "buffer_m": 1500, "note": ""},
    "Titik_07": {"coord": [110.4930955, -5.8154436], "buffer_m": 1500, "note": "erosion signal"},
    "Titik_09": {"coord": [110.4674130, -5.8075204], "buffer_m": 1500, "note": ""},
    "Titik_10": {"coord": [110.4832327, -5.8281419], "buffer_m": 1500, "note": ""},
    "Titik_11": {"coord": [110.4684736, -5.8309304], "buffer_m": 1500, "note": ""},
}
# Titik_19 sengaja tidak dimasukkan (dikeluarkan dari training set, lihat catatan proyek)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [8]:
# ============================================================
# CELL 2 — Union bounding box + fetch composite SEKALI, crop banyak AOI
# ============================================================

def get_union_bbox(aoi_configs, pad_deg=0.02):
    lons = [c['coord'][0] for c in aoi_configs.values()]
    lats = [c['coord'][1] for c in aoi_configs.values()]
    return ee.Geometry.Rectangle([
        min(lons)-pad_deg, min(lats)-pad_deg, max(lons)+pad_deg, max(lats)+pad_deg
    ])

UNION_BBOX = get_union_bbox(AOI_CONFIG)
print("Union bbox dibuat, mencakup", len(AOI_CONFIG), "AOI")


# ---------- band mapping per collection ----------
LANDSAT_BANDS = {
    'LANDSAT/LC08/C02/T1_L2': {  # Landsat 8, 2013+
        'green': 'SR_B3', 'swir1': 'SR_B6', 'qa': 'QA_PIXEL',
        'scale_factor': 0.0000275, 'offset': -0.2
    },
    'LANDSAT/LE07/C02/T1_L2': {  # Landsat 7, 2008-2012 (SLC-off, banyak gap)
        'green': 'SR_B2', 'swir1': 'SR_B5', 'qa': 'QA_PIXEL',
        'scale_factor': 0.0000275, 'offset': -0.2
    }
}

def mask_landsat_clouds(img, collection_id):
    """QA_PIXEL Collection 2: bit3=cloud, bit4=cloud shadow, bit1=dilated cloud."""
    qa_band = LANDSAT_BANDS[collection_id]['qa']
    qa = img.select(qa_band)
    cloud = qa.bitwiseAnd(1 << 3).neq(0)
    shadow = qa.bitwiseAnd(1 << 4).neq(0)
    dilated = qa.bitwiseAnd(1 << 1).neq(0)
    mask = cloud.Or(shadow).Or(dilated).Not()
    return img.updateMask(mask)


def get_landsat_collection_id(year):
    return 'LANDSAT/LC08/C02/T1_L2' if year >= 2013 else 'LANDSAT/LE07/C02/T1_L2'


def fetch_landsat_composite_once(union_geom, year, season_months, cloud_max=40):
    """SATU query per (tahun, musim) untuk SELURUH area union.
    Dipanggil sekali, hasilnya di-crop berkali-kali per AOI (Cell 3+).
    season_months: tuple ('MM-DD', 'MM-DD')"""
    collection_id = get_landsat_collection_id(year)
    bands = LANDSAT_BANDS[collection_id]
    start, end = f"{year}-{season_months[0]}", f"{year}-{season_months[1]}"

    coll = (ee.ImageCollection(collection_id)
            .filterBounds(union_geom)
            .filterDate(start, end)
            .filter(ee.Filter.lte('CLOUD_COVER', cloud_max)))
    n = coll.size().getInfo()
    if n == 0:
        return None, 0, collection_id

    def scale_bands(img):
        optical = img.select(['SR_B.*']).multiply(bands['scale_factor']).add(bands['offset'])
        return img.addBands(optical, None, True)

    coll_masked = coll.map(lambda img: mask_landsat_clouds(scale_bands(img), collection_id))
    composite = coll_masked.median().clip(union_geom)
    return composite, n, collection_id

Union bbox dibuat, mencakup 8 AOI


In [ ]:
# ============================================================
# CELL 3 — ShorelineProcessorLandsat: per-AOI crop + MNDWI + threshold
# ============================================================

class ShorelineProcessorLandsat:
    """Mirip ShorelineProcessor256, tapi:
    - collection Landsat (band beda per sensor)
    - resample 30m->10m via bilinear sebelum MNDWI
    - threshold Otsu DIPISAH dari Sentinel-2 (histogram beda populasi)
    - crop dari composite  yang sudah di-fetch sekali (Cell 2),
      BUKAN query GEE baru per AOI"""

    def __init__(self, aoi_name, lon, lat, buffer_m=1500):
        self.aoi_name = aoi_name
        self.lon, self.lat = lon, lat
        self.buffer_m = buffer_m
        self.aoi = ee.Geometry.Point([lon, lat]).buffer(buffer_m)
        self.threshold = None
        self.threshold_val = None
        self.frames = {}       # {(year, 'L1'/'L2'): {'water': ee.Image, 'n_scene': int}}

    def _add_mndwi_landsat(self, composite, collection_id):
        bands = LANDSAT_BANDS[collection_id]
        green = composite.select(bands['green'])
        swir1 = (composite.select(bands['swir1'])
                 .resample('bilinear')
                 .reproject(crs=green.projection(), scale=SCALE_M))
        mndwi = green.subtract(swir1).divide(green.add(swir1)).rename('MNDWI')
        return mndwi

    def crop_and_compute_mndwi(self, union_composite, collection_id):
        cropped = union_composite.clip(self.aoi)
        return self._add_mndwi_landsat(cropped, collection_id)

    @staticmethod
    def _otsu(histogram):
        counts = ee.Array(ee.Dictionary(histogram).get('histogram'))
        means  = ee.Array(ee.Dictionary(histogram).get('bucketMeans'))
        size   = means.length().get([0])
        total  = counts.reduce(ee.Reducer.sum(), [0]).get([0])
        sum_   = means.multiply(counts).reduce(ee.Reducer.sum(), [0]).get([0])
        mean   = sum_.divide(total)
        def bss(i):
            a_counts = counts.slice(0, 0, i)
            a_count  = a_counts.reduce(ee.Reducer.sum(), [0]).get([0])
            a_means  = means.slice(0, 0, i)
            a_mean   = (a_means.multiply(a_counts).reduce(ee.Reducer.sum(), [0])
                        .get([0]).divide(a_count))
            b_count  = total.subtract(a_count)
            b_mean   = sum_.subtract(a_count.multiply(a_mean)).divide(b_count)
            return (a_count.multiply(a_mean.subtract(mean).pow(2))
                    .add(b_count.multiply(b_mean.subtract(mean).pow(2))))
        bss_values = ee.List.sequence(1, size).map(lambda i: bss(ee.Number(i)))
        return means.sort(bss_values).get([-1])

    def fit_threshold(self, ref_mndwi, verbose=True):
        """Threshold Otsu TERPISAH untuk Landsat — dengan guard ganda:
        (1) cek jumlah pixel valid, (2) cek histogram beneran punya isi
        sebelum dilempar ke _otsu (kasus n_valid>10 tapi histogram tetap
        kosong bisa terjadi kalau semua pixel bernilai identik/no variance)."""

        valid_check = ref_mndwi.select('MNDWI').reduceRegion(
            reducer=ee.Reducer.count(),
            geometry=self.aoi, scale=SCALE_M, maxPixels=1e9
        ).get('MNDWI')
        n_valid = valid_check.getInfo()

        if not n_valid or n_valid < 10:
            print(f"⚠️  [{self.aoi_name}] Cuma {n_valid} pixel valid — SKIP.")
            self.threshold = None
            self.threshold_val = None
            return None

        hist_raw = ref_mndwi.select('MNDWI').reduceRegion(
            reducer=ee.Reducer.histogram(255, 0.01),
            geometry=self.aoi, scale=SCALE_M, maxPixels=1e9).get('MNDWI')
        hist_dict = hist_raw.getInfo()   # tarik ke client SEBELUM ke _otsu

        if not hist_dict or 'bucketMeans' not in hist_dict:
            print(f"⚠️  [{self.aoi_name}] n_valid={n_valid} tapi histogram kosong "
                f"(kemungkinan semua pixel bernilai identik). SKIP.")
            self.threshold = None
            self.threshold_val = None
            return None

        self.threshold = self._otsu(hist_raw)
        self.threshold_val = self.threshold.getInfo()
        if verbose:
            print(f"[{self.aoi_name}] Landsat threshold Otsu = {self.threshold_val:+.4f} "
                f"({n_valid} pixel valid)")
        return self.threshold_val

    def _water_mask(self, mndwi_img, closing_radius=2):
        water = mndwi_img.select('MNDWI').gt(self.threshold)
        proj = mndwi_img.select('MNDWI').projection()
        return (water
                .focalMode(radius=1, kernelType='square', units='pixels')
                .focalMax(closing_radius, kernelType='square', units='pixels')
                .focalMin(closing_radius, kernelType='square', units='pixels')
                .reproject(crs=proj, scale=SCALE_M)
                .rename('water'))

    def _keep_ocean(self, water, seed_m=100, max_dist=5000):
        aoi_mask = ee.Image.constant(1).clip(self.aoi).mask()
        inner = aoi_mask.focalMin(radius=seed_m, units='meters')
        ring = aoi_mask.subtract(inner)
        seed = water.multiply(ring).selfMask()
        cost = water.Not().multiply(1e6).add(1)
        reach = cost.cumulativeCost(source=seed, maxDistance=max_dist)
        return water.updateMask(reach.lt(1e5)).unmask(0).rename('water')

In [19]:
# ============================================================
# CELL 4 — 2 musim/tahun untuk Landsat, orkestrasi fetch-sekali-crop-banyak
# ============================================================
LANDSAT_SEASONS = {
    'L1': ('01-01', '06-30'),
    'L2': ('07-01', '12-31'),
}
LANDSAT_YEARS = list(range(2008, 2019))  # 2008-2018, nyambung ke Sentinel-2 2019+

def run_landsat_pipeline(aoi_configs, union_geom, years=LANDSAT_YEARS,
                         seasons=LANDSAT_SEASONS, ref_year=2016,
                         cloud_max=40, cooldown_s=5):
    processors = {nama: ShorelineProcessorLandsat(nama, info['coord'][0], info['coord'][1],
                                                  info.get('buffer_m', 1500))
                 for nama, info in aoi_configs.items()}

    # ---------- threshold referensi: fetch composite tahunan penuh SEKALI ----------
    ref_collection_id = get_landsat_collection_id(ref_year)
    ref_composite, ref_n, _ = fetch_landsat_composite_once(
        union_geom, ref_year, ('01-01', '12-31'), cloud_max)
    if ref_composite is None:
        raise RuntimeError(f"Tidak ada data Landsat untuk ref_year={ref_year}")
    print(f"Referensi: {ref_year}, {ref_n} scene (union)")

    for nama, p in processors.items():
        ref_mndwi = p.crop_and_compute_mndwi(ref_composite, ref_collection_id)
        p.fit_threshold(ref_mndwi)
        time.sleep(1)

    # filter processor yang threshold-nya gagal
    processors = {nama: p for nama, p in processors.items() if p.threshold_val is not None}
    print(f"AOI dengan threshold valid: {list(processors.keys())} ({len(processors)}/{len(AOI_CONFIG)})")
    # ---------- loop tahun x musim: FETCH SEKALI per slot, crop ke semua AOI ----------
    meta_log = []
    for yr in years:
        for s_name, months in seasons.items():
            composite, n, coll_id = fetch_landsat_composite_once(
                union_geom, yr, months, cloud_max)

            if composite is None or n == 0:
                print(f"{yr}-{s_name}: 0 scene -> SKIP semua AOI")
                meta_log.append({'year': yr, 'season': s_name, 'n_scene': 0, 'status': 'empty'})
                continue

            for nama, p in processors.items():
                mndwi = p.crop_and_compute_mndwi(composite, coll_id)
                water_raw = p._water_mask(mndwi)
                water = p._keep_ocean(water_raw)
                p.frames[(yr, s_name)] = {'water': water, 'n_scene': n}

            print(f"{yr}-{s_name}: {n} scene (union) -> di-crop ke {len(processors)} AOI")
            meta_log.append({'year': yr, 'season': s_name, 'n_scene': n, 'status': 'ok'})
            time.sleep(cooldown_s)  # jeda antar SLOT, bukan antar AOI -> jauh lebih sedikit request

    return processors, pd.DataFrame(meta_log)


processors_landsat, df_meta_landsat = run_landsat_pipeline(AOI_CONFIG, UNION_BBOX)
print(df_meta_landsat)

Referensi: 2016, 9 scene (union)
⚠️  [Titik_02] Cuma 0 pixel valid — SKIP.
⚠️  [Titik_04] Cuma 0 pixel valid — SKIP.
[Titik_05] Landsat threshold Otsu = +0.5950 (70347 pixel valid)
⚠️  [Titik_06] Cuma 0 pixel valid — SKIP.
[Titik_07] Landsat threshold Otsu = +0.5441 (70356 pixel valid)
[Titik_09] Landsat threshold Otsu = +0.6552 (70376 pixel valid)
⚠️  [Titik_10] Cuma 0 pixel valid — SKIP.
⚠️  [Titik_11] Cuma 0 pixel valid — SKIP.
AOI dengan threshold valid: ['Titik_05', 'Titik_07', 'Titik_09'] (3/8)
2008-L1: 5 scene (union) -> di-crop ke 3 AOI
2008-L2: 4 scene (union) -> di-crop ke 3 AOI
2009-L1: 5 scene (union) -> di-crop ke 3 AOI
2009-L2: 3 scene (union) -> di-crop ke 3 AOI
2010-L1: 3 scene (union) -> di-crop ke 3 AOI
2010-L2: 1 scene (union) -> di-crop ke 3 AOI
2011-L1: 2 scene (union) -> di-crop ke 3 AOI
2011-L2: 5 scene (union) -> di-crop ke 3 AOI
2012-L1: 4 scene (union) -> di-crop ke 3 AOI
2012-L2: 1 scene (union) -> di-crop ke 3 AOI
2013-L1: 2 scene (union) -> di-crop ke 3 AOI

In [20]:
for nama, info in AOI_CONFIG.items():
    lon, lat = info['coord']
    pt = ee.Geometry.Point([lon, lat])
    print(nama, UNION_BBOX.contains(pt).getInfo())

Titik_02 True
Titik_04 True
Titik_05 True
Titik_06 True
Titik_07 True
Titik_09 True
Titik_10 True
Titik_11 True


In [21]:
# ============================================================
# CELL 5 — Export ke .npy dengan retry+backoff (pola sama seperti sebelumnya)
# ============================================================
def _fit_patch(mask_arr, patch_size=PATCH_SIZE):
    t = patch_size
    h, w = mask_arr.shape
    if h > t: mask_arr = mask_arr[(h-t)//2:(h-t)//2+t, :]
    if w > t: mask_arr = mask_arr[:, (w-t)//2:(w-t)//2+t]
    h, w = mask_arr.shape
    if h < t or w < t:
        pad = np.zeros((t, t), dtype=mask_arr.dtype)
        pad[:h, :w] = mask_arr
        mask_arr = pad
    return mask_arr

def export_landsat_all(processors, out_dir=LANDSAT_MASK_DIR, max_retry=3, cooldown_s=3):
    masks = {}
    failed_jobs = []
    for nama, p in processors.items():
        aoi_dir = os.path.join(out_dir, nama)
        os.makedirs(aoi_dir, exist_ok=True)
        masks[nama] = {}
        for (yr, s), f in sorted(p.frames.items()):
            path = os.path.join(aoi_dir, f"{yr}_{s}.npy")
            if os.path.exists(path):
                masks[nama][(yr, s)] = np.load(path)
                continue
            last_err = None
            for attempt in range(max_retry):
                try:
                    region = ee.Geometry.Point([p.lon, p.lat]).buffer(PATCH_SIZE*SCALE_M/2).bounds()
                    url = f['water'].getDownloadURL({'region': region, 'scale': SCALE_M, 'format': 'NPY'})
                    r = requests.get(url, timeout=60)
                    r.raise_for_status()
                    m = _fit_patch(np.load(io.BytesIO(r.content))['water'].astype(np.float32))
                    np.save(path, m)
                    masks[nama][(yr, s)] = m
                    time.sleep(cooldown_s)
                    break
                except Exception as e:
                    last_err = e
                    if attempt < max_retry - 1:
                        time.sleep(5 * (attempt+1))
            else:
                failed_jobs.append((nama, (yr, s), str(last_err)))
        print(f"[{nama}] {len(masks[nama])} frame ter-export")
    return masks, failed_jobs

masks_landsat, failed_landsat = export_landsat_all(processors_landsat)
print(f"Gagal: {failed_landsat}")

[Titik_05] 22 frame ter-export
[Titik_07] 22 frame ter-export
[Titik_09] 22 frame ter-export
Gagal: []


In [22]:
# 1. Config cuma buat titik yang gagal, sudah dilonggarkan
AOI_CONFIG_RETRY = {
    nama: {**info, 'buffer_m': info.get('buffer_m', 1500) * 2}  # contoh: buffer 2x lipat
    for nama, info in AOI_CONFIG.items()
    if nama not in processors_landsat  # cuma yang tadinya ke-skip
}
print("Retry buat:", list(AOI_CONFIG_RETRY.keys()))

# 2. Jalanin pipeline CUMA buat subset ini, cloud_max lebih longgar misalnya
processors_retry, df_meta_retry = run_landsat_pipeline(
    AOI_CONFIG_RETRY, UNION_BBOX, cloud_max=60
)

# 3. Gabungin ke hasil lama (bukan nimpa)
processors_landsat.update(processors_retry)
df_meta_landsat = pd.concat([df_meta_landsat, df_meta_retry], ignore_index=True)

print("Total AOI sekarang:", list(processors_landsat.keys()))

Retry buat: ['Titik_02', 'Titik_04', 'Titik_06', 'Titik_10', 'Titik_11']
Referensi: 2016, 11 scene (union)
[Titik_02] Landsat threshold Otsu = +0.4553 (281426 pixel valid)
⚠️  [Titik_04] Cuma 0 pixel valid — SKIP.
[Titik_06] Landsat threshold Otsu = +0.4650 (281438 pixel valid)
[Titik_10] Landsat threshold Otsu = +0.3853 (272077 pixel valid)
[Titik_11] Landsat threshold Otsu = +0.4754 (260952 pixel valid)
AOI dengan threshold valid: ['Titik_02', 'Titik_06', 'Titik_10', 'Titik_11'] (4/8)
2008-L1: 5 scene (union) -> di-crop ke 4 AOI
2008-L2: 4 scene (union) -> di-crop ke 4 AOI
2009-L1: 5 scene (union) -> di-crop ke 4 AOI
2009-L2: 4 scene (union) -> di-crop ke 4 AOI
2010-L1: 5 scene (union) -> di-crop ke 4 AOI
2010-L2: 2 scene (union) -> di-crop ke 4 AOI
2011-L1: 4 scene (union) -> di-crop ke 4 AOI
2011-L2: 5 scene (union) -> di-crop ke 4 AOI
2012-L1: 7 scene (union) -> di-crop ke 4 AOI
2012-L2: 2 scene (union) -> di-crop ke 4 AOI
2013-L1: 2 scene (union) -> di-crop ke 4 AOI
2013-L2: 6 sc

In [23]:
AOI_CONFIG_T04 = {
    'Titik_04': {
        **AOI_CONFIG['Titik_04'],
        'buffer_m': AOI_CONFIG['Titik_04'].get('buffer_m', 1500) * 4,  # 4x dari awal
    }
}
processors_t04, df_t04 = run_landsat_pipeline(AOI_CONFIG_T04, UNION_BBOX, cloud_max=80)

Referensi: 2016, 14 scene (union)
[Titik_04] Landsat threshold Otsu = +0.3152 (688558 pixel valid)
AOI dengan threshold valid: ['Titik_04'] (1/8)
2008-L1: 5 scene (union) -> di-crop ke 1 AOI
2008-L2: 4 scene (union) -> di-crop ke 1 AOI
2009-L1: 6 scene (union) -> di-crop ke 1 AOI
2009-L2: 4 scene (union) -> di-crop ke 1 AOI
2010-L1: 5 scene (union) -> di-crop ke 1 AOI
2010-L2: 2 scene (union) -> di-crop ke 1 AOI
2011-L1: 4 scene (union) -> di-crop ke 1 AOI
2011-L2: 6 scene (union) -> di-crop ke 1 AOI
2012-L1: 7 scene (union) -> di-crop ke 1 AOI
2012-L2: 5 scene (union) -> di-crop ke 1 AOI
2013-L1: 3 scene (union) -> di-crop ke 1 AOI
2013-L2: 7 scene (union) -> di-crop ke 1 AOI
2014-L1: 8 scene (union) -> di-crop ke 1 AOI
2014-L2: 8 scene (union) -> di-crop ke 1 AOI
2015-L1: 9 scene (union) -> di-crop ke 1 AOI
2015-L2: 10 scene (union) -> di-crop ke 1 AOI
2016-L1: 5 scene (union) -> di-crop ke 1 AOI
2016-L2: 9 scene (union) -> di-crop ke 1 AOI
2017-L1: 8 scene (union) -> di-crop ke 1 AO

In [24]:
# gabung Titik_04 ke processor utama
processors_landsat.update(processors_t04)
print("Total AOI sekarang:", list(processors_landsat.keys()))

Total AOI sekarang: ['Titik_05', 'Titik_07', 'Titik_09', 'Titik_02', 'Titik_06', 'Titik_10', 'Titik_11', 'Titik_04']


In [16]:
!find /content/drive -type d -iname "*landsat*mask*" 2>/dev/null
!find /content -maxdepth 4 -type d -iname "*landsat*" 2>/dev/null

/content/drive/MyDrive/Data_experiment_shoreline/masks_landsat


In [25]:
LANDSAT_MASK_DIR = "/content/drive/MyDrive/Data_experiment_shoreline/masks_landsat"

In [ ]:
def export_landsat_all(processors, out_dir=LANDSAT_MASK_DIR, max_retry=3, cooldown_s=1):
    masks = {}
    failed_jobs = []
    t_start = time.time()
    for nama, p in processors.items():
        aoi_dir = os.path.join(out_dir, nama)
        os.makedirs(aoi_dir, exist_ok=True)
        masks[nama] = {}
        for (yr, s), f in sorted(p.frames.items()):
            path = os.path.join(aoi_dir, f"{yr}_{s}.npy")
            if os.path.exists(path):
                masks[nama][(yr, s)] = np.load(path)
                continue
            t0 = time.time()
            print(f"  [{time.time()-t_start:5.0f}s] fetching {nama} {yr}-{s}...", end=" ", flush=True)
            last_err = None
            for attempt in range(max_retry):
                try:
                    region = ee.Geometry.Point([p.lon, p.lat]).buffer(PATCH_SIZE*SCALE_M/2).bounds()
                    url = f['water'].getDownloadURL({'region': region, 'scale': SCALE_M, 'format': 'NPY'})
                    r = requests.get(url, timeout=60)
                    r.raise_for_status()
                    m = _fit_patch(np.load(io.BytesIO(r.content))['water'].astype(np.float32))
                    np.save(path, m)
                    masks[nama][(yr, s)] = m
                    print(f"OK ({time.time()-t0:.1f}s)")
                    time.sleep(cooldown_s)
                    break
                except Exception as e:
                    last_err = e
                    print(f"retry({type(e).__name__})", end=" ", flush=True)
                    if attempt < max_retry - 1:
                        time.sleep(5 * (attempt+1))
            else:
                print("GAGAL")
                failed_jobs.append((nama, (yr, s), str(last_err)))
        print(f"[{nama}] {(masks[nama])} frame ter-export")
    return masks, failed_jobs

masks_landsat, failed_landsat = export_landsat_all(processors_landsat, cooldown_s=1)
print(f"Gagal: {failed_landsat}")

[Titik_05] 22 frame ter-export
[Titik_07] 22 frame ter-export
[Titik_09] 22 frame ter-export
  [    0s] fetching Titik_02 2016-L1... OK (36.4s)
[Titik_02] 22 frame ter-export
  [   38s] fetching Titik_06 2009-L1... OK (19.6s)
[Titik_06] 22 frame ter-export
  [   58s] fetching Titik_10 2008-L1... OK (18.6s)
  [   78s] fetching Titik_10 2008-L2... OK (32.6s)
  [  112s] fetching Titik_10 2009-L1... OK (19.0s)
  [  132s] fetching Titik_10 2009-L2... OK (39.8s)
  [  172s] fetching Titik_10 2010-L1... OK (48.4s)
  [  222s] fetching Titik_10 2010-L2... OK (18.8s)
  [  242s] fetching Titik_10 2011-L1... retry(ReadTimeout) OK (83.1s)
  [  326s] fetching Titik_10 2011-L2... retry(ReadTimeout) OK (84.8s)
  [  411s] fetching Titik_10 2012-L1... OK (26.4s)
  [  439s] fetching Titik_10 2012-L2... retry(ReadTimeout) OK (103.9s)
  [  544s] fetching Titik_10 2014-L1... OK (23.0s)
  [  568s] fetching Titik_10 2014-L2... OK (20.2s)
  [  589s] fetching Titik_10 2015-L1... OK (20.2s)
  [  610s] fetching Ti

OK (20.2s)
  [ 1842s] fetching Titik_04 2008-L2... retry(ReadTimeout) OK (108.5s)
  [ 1951s] fetching Titik_04 2009-L1... retry(ReadTimeout) retry(ReadTimeout) OK (164.6s)
  [ 2117s] fetching Titik_04 2011-L2... retry(ReadTimeout) OK (87.2s)
  [ 2205s] fetching Titik_04 2012-L2... retry(ReadTimeout) OK (99.8s)
  [ 2306s] fetching Titik_04 2013-L2... OK (32.7s)
  [ 2339s] fetching Titik_04 2014-L1... OK (48.2s)
  [ 2389s] fetching Titik_04 2014-L2... OK (18.9s)
  [ 2408s] fetching Titik_04 2015-L1... OK (21.2s)
  [ 2431s] fetching Titik_04 2015-L2... OK (28.9s)
  [ 2460s] fetching Titik_04 2016-L1... OK (46.9s)
  [ 2508s] fetching Titik_04 2016-L2... OK (21.4s)
  [ 2531s] fetching Titik_04 2017-L1... OK (20.3s)
  [ 2552s] fetching Titik_04 2017-L2... retry(ReadTimeout) OK (84.2s)
  [ 2637s] fetching Titik_04 2018-L1... OK (18.7s)
  [ 2657s] fetching Titik_04 2018-L2... OK (25.5s)
[Titik_04] 22 frame ter-export
Gagal: []


In [28]:
import shutil

def export_offline_landsat(processors, masks, failed_jobs=None,
                            out_dir="/content/drive/MyDrive/shoreline_kemujan/landsat_offline",
                            bundle=True):
    os.makedirs(out_dir, exist_ok=True)
    failed_jobs = failed_jobs or []
    manifest = {
        "created_utc": pd.Timestamp.utcnow().isoformat(),
        "gee_project": GEE_PROJECT,
        "source": {
            "sensor": "Landsat (union composite, 30m)",
            "scale_m": SCALE_M,
            "note": "Landsat = konteks visual/kualitatif, TIDAK dipakai sebagai input ConvLSTM utama.",
        },
        "params": {
            "PATCH_SIZE": PATCH_SIZE,
            "LANDSAT_SEASONS": LANDSAT_SEASONS,
            "LANDSAT_YEARS": LANDSAT_YEARS,
        },
        "aoi": {},
        "failed_jobs": [
            {"aoi": nama, "year": yr, "season": s, "error": err}
            for nama, (yr, s), err in failed_jobs
        ],
    }
    for nama, p in processors.items():
        aoi_entry = {"lon": p.lon, "lat": p.lat, "buffer_m": getattr(p, "buffer_m", None)}
        if hasattr(p, "threshold_val"):
            aoi_entry["threshold_val"] = p.threshold_val
        aoi_entry["frames"] = {
            f"{k[0]}_{k[1]}": v.get("n_scene") if isinstance(v, dict) else None
            for k, v in sorted(p.frames.items())
        }
        manifest["aoi"][nama] = aoi_entry
    with open(os.path.join(out_dir, "manifest.json"), "w") as f:
        json.dump(manifest, f, indent=2)
    for nama, frames in masks.items():
        arrays = {f"{k[0]}_{k[1]}": v for k, v in frames.items()}
        np.savez_compressed(os.path.join(out_dir, f"{nama}_landsat_masks.npz"), **arrays)
    rows = []
    for nama, p in processors.items():
        for (yr, s), fr in sorted(p.frames.items()):
            n_scene = fr.get("n_scene") if isinstance(fr, dict) else None
            rows.append({"aoi": nama, "year": yr, "season": s,
                        "n_scene": n_scene, "exported": (yr, s) in masks.get(nama, {})})
    pd.DataFrame(rows).to_csv(os.path.join(out_dir, "frames_summary.csv"), index=False)
    size_mb = sum(os.path.getsize(os.path.join(out_dir, f))
                  for f in os.listdir(out_dir)
                  if os.path.isfile(os.path.join(out_dir, f))) / 1e6
    print(f"Offline bundle Landsat: {out_dir} | {len(masks)} AOI | {size_mb:.1f} MB")
    if bundle:
        zip_path = shutil.make_archive(out_dir, "zip", out_dir)
        print(f"Zip siap download: {zip_path} ({os.path.getsize(zip_path)/1e6:.1f} MB)")
        return out_dir, zip_path
    return out_dir, None

In [29]:
out_dir, zip_path = export_offline_landsat(processors_landsat, masks_landsat, failed_landsat)

Offline bundle Landsat: /content/drive/MyDrive/shoreline_kemujan/landsat_offline | 8 AOI | 0.3 MB
Zip siap download: /content/drive/MyDrive/shoreline_kemujan/landsat_offline.zip (0.2 MB)


In [30]:
print(LANDSAT_MASK_DIR)
print(os.path.abspath(LANDSAT_MASK_DIR))

/content/drive/MyDrive/Data_experiment_shoreline/masks_landsat
/content/drive/MyDrive/Data_experiment_shoreline/masks_landsat


In [31]:
# ============================================================
# CELL 6 — Unified time indexing: Landsat (2 slot/th) + Sentinel-2 (3 slot/th)
# ============================================================
SEASON_ORDER_S2 = ['S1', 'S2', 'S3']
SEASON_ORDER_LANDSAT = ['L1', 'L2']

# mid-bulan tiap slot, buat sortir konsisten lintas skema musim beda cacah
SEASON_MID_MONTH = {
    'S1': 2, 'S2': 6, 'S3': 10,   # Sentinel-2: Jan-Apr, Mei-Ags, Sep-Des
    'L1': 3, 'L2': 9,             # Landsat: Jan-Jun, Jul-Des
}

def abs_month(year, season_label):
    """Konversi (tahun, label musim) -> bulan absolut (sortable, unified).
    Ganti seq_index() lama yang asumsi 3 musim seragam."""
    return year * 12 + SEASON_MID_MONTH[season_label]

def combine_landsat_sentinel(masks_landsat, masks_sentinel2):
    """Gabung frame Landsat + Sentinel-2 per AOI jadi satu dict terurut
    berdasarkan abs_month, key tetap (year, season_label) asli."""
    combined = {}
    all_aoi = set(masks_landsat.keys()) | set(masks_sentinel2.keys())
    for nama in all_aoi:
        frames = {}
        frames.update(masks_landsat.get(nama, {}))
        frames.update(masks_sentinel2.get(nama, {}))
        combined[nama] = frames
    return combined

def sorted_keys(frames):
    return sorted(frames.keys(), key=lambda k: abs_month(*k))

In [32]:
# ============================================================
# CELL 7 — QA WAJIB: overlay basemap tiap frame Landsat baru
# ============================================================
!pip install contextily -q
import contextily as ctx

def qa_landsat_frame(nama, masks_landsat, key, lon_c, lat_c, buffer_m=1500):
    mask = masks_landsat[nama].get(key)
    if mask is None:
        print(f"[{nama}] {key} tidak ada"); return
    water_pct = mask.mean() * 100

    fig, ax = plt.subplots(figsize=(6,6))
    ax.imshow(mask, cmap='Blues', vmin=0, vmax=1, interpolation='nearest')
    ax.set_title(f"{nama} {key[0]}-{key[1]} (Landsat) | {water_pct:.0f}% air")
    ax.axis('off')
    plt.show()
    return water_pct

def qa_landsat_all(masks_landsat, aoi_configs):
    """Loop QA cepat -- cek land_frac tidak collapse (0% atau 100% ekstrem)
    dan tidak kebalik arah kelas (air > 90% padahal AOI didominasi darat)."""
    rows = []
    for nama, frames in masks_landsat.items():
        for key, mask in frames.items():
            pct = mask.mean() * 100
            flag = 'uniform_water' if pct > 98 else ('uniform_land' if pct < 2 else 'ok')
            rows.append({'aoi': nama, 'year': key[0], 'season': key[1],
                        'water_pct': pct, 'flag': flag})
    df = pd.DataFrame(rows)
    print(df[df.flag != 'ok'])
    return df

qa_df_landsat = qa_landsat_all(masks_landsat, AOI_CONFIG)

          aoi  year season   water_pct           flag
4    Titik_05  2010     L1    0.000000   uniform_land
5    Titik_05  2010     L2    0.000000   uniform_land
6    Titik_05  2011     L1    0.000000   uniform_land
9    Titik_05  2012     L2    0.000000   uniform_land
12   Titik_05  2014     L1    0.000000   uniform_land
26   Titik_07  2010     L1    0.000000   uniform_land
27   Titik_07  2010     L2    0.000000   uniform_land
28   Titik_07  2011     L1    0.000000   uniform_land
31   Titik_07  2012     L2    0.000000   uniform_land
34   Titik_07  2014     L1    0.770569   uniform_land
37   Titik_07  2015     L2    0.636292   uniform_land
41   Titik_07  2017     L2    0.527954   uniform_land
48   Titik_09  2010     L1    0.000000   uniform_land
49   Titik_09  2010     L2    0.000000   uniform_land
50   Titik_09  2011     L1    0.000000   uniform_land
53   Titik_09  2012     L2    0.000000   uniform_land
54   Titik_09  2013     L1    1.274109   uniform_land
56   Titik_09  2014     L1  

In [ ]:
# ============================================================
# CELL 8 — Cek sambungan sensor 2018 (Landsat) -> 2019 (Sentinel-2)
# ============================================================
def check_sensor_transition(nama, masks_landsat, masks_sentinel2):
    """Plot pct_water time series lintas Landsat->Sentinel-2, cari
    step-function/lompatan yang nunjukin kalibrasi 2 sensor tidak nyambung."""
    landsat_frames = masks_landsat.get(nama, {})
    s2_frames = masks_sentinel2.get(nama, {})

    keys_l = sorted(landsat_frames.keys(), key=lambda k: abs_month(*k))
    keys_s = sorted(s2_frames.keys(), key=lambda k: abs_month(*k))

    x_l = [abs_month(*k) for k in keys_l]
    y_l = [landsat_frames[k].mean()*100 for k in keys_l]
    x_s = [abs_month(*k) for k in keys_s]
    y_s = [s2_frames[k].mean()*100 for k in keys_s]

    fig, ax = plt.subplots(figsize=(14, 4))
    ax.plot(x_l, y_l, 'o-', color='tab:orange', label='Landsat (2008-2018)')
    ax.plot(x_s, y_s, 'o-', color='tab:blue', label='Sentinel-2 (2019+)')
    ax.axvline(abs_month(2019, 'S1'), color='red', ls='--', alpha=0.6, label='Transisi sensor')
    ax.set_xlabel("Bulan absolut"); ax.set_ylabel("% air (pct_water)")
    ax.set_title(f"{nama} — Cek sambungan sensor Landsat -> Sentinel-2")
    ax.legend()
    plt.tight_layout(); plt.show()

    if y_l and y_s:
        gap = abs(y_l[-1] - y_s[0])
        print(f"[{nama}] Selisih pct_water di titik sambungan: {gap:.1f} poin persen")
        if gap > 15:
            print(f"⚠️  Selisih besar -- indikasi kalibrasi 2 sensor tidak mulus, "
                 f"pertimbangkan koreksi bias sebelum digabung ke training")